## 📦 Part 1: Environment Setup & Dependencies

### Install Required Packages

In [ ]:
# Install core dependencies
# Run this cell only once

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install transformers accelerate
!pip install opencv-python pillow
!pip install facenet-pytorch
!pip install matplotlib seaborn
!pip install tqdm

### Import All Required Libraries

In [ ]:
# Standard library imports
import sys
import os
import json
import warnings
from pathlib import Path
from datetime import timedelta
from collections import Counter
from typing import List, Dict, Tuple, Optional, Any

# Data science imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Computer vision imports
import cv2
from PIL import Image

# Deep learning imports
import torch
from transformers import (
    LlavaNextProcessor, 
    LlavaNextForConditionalGeneration,
    AutoImageProcessor,
    AutoModelForImageClassification
)
from facenet_pytorch import MTCNN, InceptionResnetV1

# Suppress warnings
warnings.filterwarnings('ignore')

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"📦 PyTorch version: {torch.__version__}")
print(f"🖥️  Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

---

## 🎬 Part 2: Keyframe Extraction Module

### Theory: Video Frame Extraction

**What is a Keyframe?**
- A representative frame extracted from a video at specified time intervals
- Used to reduce computational cost while maintaining temporal coverage
- Example: For a 60-second video at 30 FPS (1800 frames), extracting 1 frame/second gives 60 keyframes (97% reduction)

**OpenCV VideoCapture Workflow:**
1. Open video file and read metadata (FPS, resolution, duration)
2. Calculate frame interval based on desired time interval
3. Read frames sequentially and extract at intervals
4. Convert color space (BGR → RGB) for PIL/PyTorch compatibility
5. Store frame data with timestamp metadata

**Key Formulas:**
- `frame_interval = FPS × interval_seconds`
- `timestamp = frame_number / FPS`
- `total_frames = video_duration × FPS`

In [ ]:
def extract_keyframes(
    video_path: str,
    interval_sec: float = 1.0,
    max_frames: Optional[int] = None
) -> List[Dict[str, Any]]:
    """
    Extract keyframes from video at specified time intervals.
    
    Args:
        video_path (str): Path to input video file
        interval_sec (float): Time interval between keyframes in seconds
        max_frames (int, optional): Maximum number of frames to extract
    
    Returns:
        List[Dict]: List of keyframe dictionaries containing:
            - frame (PIL.Image): The extracted frame as RGB PIL Image
            - timestamp (float): Time position in seconds
            - frame_number (int): Frame index in original video
            - formatted_time (str): Human-readable time (HH:MM:SS)
            - fps (float): Video frames per second
    
    Process:
        1. Validate video file exists
        2. Open video with cv2.VideoCapture
        3. Read video metadata (FPS, resolution, duration)
        4. Calculate frame interval (frames to skip)
        5. Loop through video frames:
           - Check if frame_count % frame_interval == 0
           - If yes: extract, convert BGR→RGB, create PIL Image
           - Calculate timestamp and formatted time
           - Store in keyframes list
        6. Release video capture and return keyframes
    """
    
    # ========================================================================
    # STEP 1: VALIDATE INPUT
    # ========================================================================
    
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Video file not found: {video_path}")
    
    print(f"\n{'='*80}")
    print(f"📹 EXTRACTING KEYFRAMES FROM VIDEO")
    print(f"{'='*80}")
    print(f"📁 File: {video_path.name}")
    
    # ========================================================================
    # STEP 2: OPEN VIDEO AND READ METADATA
    # ========================================================================
    
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        raise ValueError(f"Failed to open video: {video_path}")
    
    # Get video properties using OpenCV property IDs
    fps = cap.get(cv2.CAP_PROP_FPS)                    # Frames per second
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # Total frame count
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))    # Frame width (pixels)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))  # Frame height (pixels)
    duration = total_frames / fps if fps > 0 else 0   # Duration (seconds)
    
    print(f"\n📊 Video Properties:")
    print(f"   Resolution:    {width} × {height} pixels")
    print(f"   FPS:           {fps:.2f} frames/second")
    print(f"   Total Frames:  {total_frames:,}")
    print(f"   Duration:      {duration:.2f} seconds ({timedelta(seconds=int(duration))})")
    
    # ========================================================================
    # STEP 3: CALCULATE FRAME EXTRACTION PARAMETERS
    # ========================================================================
    
    # Calculate how many frames to skip between keyframes
    # Example: 30 FPS × 2.0 seconds = extract every 60th frame
    frame_interval = int(fps * interval_sec)
    if frame_interval < 1:
        frame_interval = 1
    
    # Estimate number of keyframes
    expected_frames = int(duration / interval_sec)
    
    print(f"\n🎯 Extraction Settings:")
    print(f"   Interval:         {interval_sec}s per keyframe")
    print(f"   Frame Interval:   Every {frame_interval} frames")
    print(f"   Expected Output:  ~{expected_frames} keyframes")
    if max_frames:
        print(f"   Max Limit:        {max_frames} frames")
    
    # ========================================================================
    # STEP 4: EXTRACT KEYFRAMES
    # ========================================================================
    
    keyframes = []
    frame_count = 0
    extracted_count = 0
    
    print(f"\n⏳ Extracting keyframes...")
    progress_bar = tqdm(total=expected_frames, desc="Progress", unit="frames")
    
    try:
        while cap.isOpened():
            # Read next frame
            ret, frame = cap.read()
            
            # Check if frame was read successfully
            if not ret:
                break  # End of video
            
            # Check if this frame should be extracted
            if frame_count % frame_interval == 0:
                # Convert BGR (OpenCV default) to RGB (standard format)
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                
                # Convert NumPy array to PIL Image
                pil_image = Image.fromarray(frame_rgb)
                
                # Calculate timestamp (frame position in seconds)
                timestamp = frame_count / fps if fps > 0 else 0
                
                # Format timestamp as HH:MM:SS
                time_delta = timedelta(seconds=timestamp)
                formatted_time = str(time_delta).split('.')[0]
                
                # Store keyframe data
                keyframe_data = {
                    'frame': pil_image,
                    'timestamp': timestamp,
                    'frame_number': frame_count,
                    'formatted_time': formatted_time,
                    'fps': fps
                }
                
                keyframes.append(keyframe_data)
                extracted_count += 1
                progress_bar.update(1)
                
                # Check if we've reached the maximum
                if max_frames and extracted_count >= max_frames:
                    print(f"\n⚠️  Reached maximum frame limit: {max_frames}")
                    break
            
            frame_count += 1
    
    finally:
        # Always release video capture
        cap.release()
        progress_bar.close()
    
    # ========================================================================
    # STEP 5: VALIDATE AND RETURN
    # ========================================================================
    
    if len(keyframes) == 0:
        raise ValueError("No frames could be extracted from video")
    
    print(f"\n✅ Successfully extracted {len(keyframes)} keyframes")
    print(f"   Time range: {keyframes[0]['formatted_time']} to {keyframes[-1]['formatted_time']}")
    print(f"{'='*80}\n")
    
    return keyframes


print("✅ Keyframe extraction function defined")

### Test Keyframe Extraction

**Before running:** Place a sample video file in the project directory and update the path below.

In [ ]:
# Configure test video path
TEST_VIDEO_PATH = "sample_video.mp4"  # ⚠️ CHANGE THIS TO YOUR VIDEO PATH

# Check if video exists
if not Path(TEST_VIDEO_PATH).exists():
    print(f"⚠️  Video not found: {TEST_VIDEO_PATH}")
    print("\n💡 To continue:")
    print("   1. Place a video file in the project directory")
    print("   2. Update TEST_VIDEO_PATH variable above")
    print("   3. Re-run this cell")
else:
    # Extract keyframes
    keyframes = extract_keyframes(
        video_path=TEST_VIDEO_PATH,
        interval_sec=2.0,  # Extract 1 frame every 2 seconds
        max_frames=10      # Limit to 10 frames for quick testing
    )
    
    # Display first keyframe
    plt.figure(figsize=(12, 6))
    plt.imshow(keyframes[0]['frame'])
    plt.title(f"First Keyframe (t={keyframes[0]['formatted_time']})")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Extracted {len(keyframes)} keyframes for testing")